In [ ]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Latihan

Kerjakan dengan tangan dulu. Notebook ini menyiapkan langkahnya. Jawaban yang sudah dicek ada di `latihan-kunci.ipynb`.

Indeks adalah nomor posisi, mulai dari 0.

## Latihan 1

Urutan contoh: indeks 0 sisihkan, indeks 1 campur, indeks 2 simpan.

Program Python: 0.05, 0.15, dan 0.80.
Program pembaca: 0.0499999, 0.1499997, dan 0.8000004.

1. Apakah indeks kelas yang paling besar sama?
2. Apakah selisih mutlak paling besar masih di bawah `1e-5`?


In [ ]:
# Satu ide: panggil NumPy. Pustaka adalah paket yang dipanggil dari program.
import numpy as np


In [ ]:
# Satu ide: selisih contoh kecil, dicetak supaya angkanya tidak ditulis dari ingatan.
python_kartu = np.array([0.05, 0.15, 0.80])
pembaca_kartu = np.array([0.0499999, 0.1499997, 0.8000004])
selisih = np.abs(python_kartu - pembaca_kartu)
print("selisih_contoh", float(selisih.max()))
print("batas", float(1e-5))
print("contoh_dalam_batas", bool(float(selisih.max()) <= 1e-5))
print("kelas_contoh_python", int(python_kartu.argmax()))
print("kelas_contoh_pembaca", int(pembaca_kartu.argmax()))
print("label_contoh_identik", bool(int(python_kartu.argmax()) == int(pembaca_kartu.argmax())))


In [ ]:
# Satu ide: panggil pandas, pustaka untuk tabel.
import pandas as pd


In [ ]:
# Satu ide: daftarkan folder modul ini supaya generator sintetis bisa dipanggil.
# import artinya memanggil berkas atau pustaka yang sudah ada.
# Path menunjuk ke folder atau berkas.
import sys
from pathlib import Path

folder_modul = Path.cwd()
if not (folder_modul / "tambang_sintetis.py").exists():
    folder_modul = Path.cwd() / "phase1-ml" / "modul-08-proyek-akhir"
sys.path.insert(0, str(folder_modul.resolve()))


In [ ]:
# Satu ide: panggil generator sintetis di folder modul ini.
from tambang_sintetis import NAMA_FITUR, NAMA_KELAS, buat_batch


## Latihan 2

Bangun batch sintetis, latih pipeline, ekspor ONNX dengan opset yang dikunci dan `zipmap` mati, lalu bandingkan seluruh data uji. Label kelas harus identik. Selisih probabilitas tidak boleh melebihi `atol=1e-5`.


In [ ]:
# Satu ide: buat batch sintetis. Bukan data asli perusahaan, dan tidak ada unduhan.
tabel = buat_batch(SEED)


In [ ]:
# Satu ide: cetak ukuran tabel dan nama kolom.
print("baris", int(tabel.shape[0]))
print("kolom", int(tabel.shape[1]))
print("fitur", list(NAMA_FITUR))
print("nama_kelas", list(NAMA_KELAS))


In [ ]:
# Satu ide: lihat beberapa baris pertama.
# Indeks adalah nomor baris pada tabel, mulai dari 0. Nomor itu bukan kolom kadar.
tabel.head()


## Lanjutkan latihan 2

In [ ]:
# Satu ide: panggil alat scikit-learn.
# Pipeline merangkai penyiapan angka dan model, sesuai topik modul 6.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
# Satu ide: bagi data latih dan data uji.
# Data latih dipakai model untuk belajar. Data uji disisihkan dan tidak dilihat saat belajar.
# 0.25 berarti seperempat baris menjadi data uji. stratify menjaga proporsi kelas.
X = tabel[list(NAMA_FITUR)].to_numpy()
y = tabel["kelas"].to_numpy()
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)


In [ ]:
# Satu ide: cetak banyaknya baris di tiap bagian.
print("n_latih", int(len(y_latih)))
print("n_uji", int(len(y_uji)))


In [ ]:
# Satu ide: rangkai penyiapan dan model.
# StandardScaler menggeser tiap kolom: kurang rata-rata, bagi sebaran, supaya besar kolom sebanding.
# Regresi logistik, dari modul 4, menghitung probabilitas tiap kelas.
# Probabilitas adalah angka antara 0 dan 1; pada satu baris jumlah semua kelasnya 1.
# max_iter membatasi langkah belajar supaya prosesnya berhenti.
pipeline = Pipeline(
    [
        ("siapkan", StandardScaler()),
        ("model", LogisticRegression(random_state=SEED, max_iter=500)),
    ]
)


In [ ]:
# Satu ide: model belajar hanya dari data latih.
# Scaler di dalam Pipeline ikut belajar dari data latih saja, supaya tidak terjadi data leakage.
pipeline = pipeline.fit(X_latih, y_latih)


In [ ]:
# Satu ide: siapkan angka float32 untuk berkas ONNX.
# float32 menyimpan angka lebih pendek daripada angka biasa Python, jadi selisih kecil boleh muncul.
masukan = np.ascontiguousarray(X_uji, dtype=np.float32)


In [ ]:
# Satu ide: inferensi di Python pada data uji. Model tidak dilatih ulang di sel ini.
prob_python = pipeline.predict_proba(masukan)
label_python = pipeline.predict(masukan)
print("baris_uji", int(len(label_python)))


## Ekspor

In [ ]:
# Satu ide: kunci edisi opset. Mengunci artinya nomor itu ditulis tetap.
# Angka ini yang ditulis di dokumen modul.
OPSET = 22


In [ ]:
# Satu ide: panggil skl2onnx, alat yang mengubah model scikit-learn menjadi berkas ONNX.
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


In [ ]:
# Satu ide: ekspor pipeline ke berkas ONNX.
# None berarti jumlah baris boleh berubah. Jumlah kolom mengikuti banyaknya fitur.
# Fitur adalah kolom angka yang dipakai untuk menebak.
# zipmap False meminta probabilitas sebagai tabel angka, bukan kamus nama-dan-angka.
berkas = convert_sklearn(
    pipeline,
    initial_types=[("masukan_angka", FloatTensorType([None, len(NAMA_FITUR)]))],
    target_opset=OPSET,
    options={"zipmap": False},
)


In [ ]:
# Satu ide: cetak edisi yang tertulis di berkas. Domain kosong berarti ai.onnx.
print("opset_dikunci", OPSET)
for impor in berkas.opset_import:
    nama = impor.domain if impor.domain else "ai.onnx"
    print("opset", nama, int(impor.version))


In [ ]:
# Satu ide: panggil onnxruntime, program yang membaca berkas ONNX dan menghitung tebakan.
import onnxruntime as ort


In [ ]:
# Satu ide: inferensi pada berkas ONNX, di prosesor biasa.
# None meminta semua keluaran. CPUExecutionProvider artinya tanpa kartu grafis.
sesi = ort.InferenceSession(
    berkas.SerializeToString(),
    providers=["CPUExecutionProvider"],
)
pasangan = sesi.run(None, {"masukan_angka": masukan})
keluar = {item.name: nilai for item, nilai in zip(sesi.get_outputs(), pasangan)}
label_onnx = keluar["label"]
prob_onnx = keluar["probabilities"]


In [ ]:
# Satu ide: bandingkan label dan probabilitas. Batas selisihnya atol=1e-5.
selisih_maks = float(np.max(np.abs(prob_python - prob_onnx)))
label_identik = bool(np.array_equal(label_python, label_onnx))
print("label_identik", label_identik)
print("selisih_maks", selisih_maks)
print("dalam_batas", selisih_maks <= 1e-5)


## Latihan 3

Mini-proyek ujian tingkat pro: bandingkan 8 baris pertama data uji. Aturannya sama.


In [ ]:
# Satu ide: mini-proyek ujian pada delapan baris pertama data uji.
n_mini = 8
label_mini_python = [NAMA_KELAS[int(i)] for i in label_python[:n_mini]]
label_mini_onnx = [NAMA_KELAS[int(i)] for i in label_onnx[:n_mini]]
selisih_mini = float(np.max(np.abs(prob_python[:n_mini] - prob_onnx[:n_mini])))
print("n_mini", n_mini)
print("label_mini_python", label_mini_python)
print("label_mini_onnx", label_mini_onnx)
print("label_mini_identik", label_mini_python == label_mini_onnx)
print("selisih_mini", selisih_mini)
print("mini_dalam_batas", selisih_mini <= 1e-5)
